<a href="https://colab.research.google.com/github/Todekk/AI-Agents-and-Workflows-for-Developers/blob/main/Pusheen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langgraph langchain-openai

# Code reviewer

This project implements a multi-agent workflow for software development.

The system contains two AI agents:

**Code Writer Agent**: receives a programming request and generates Python code.

**Code Reviewer Agent**: reviews the generated code, identifies possible problems, and suggests improvements.

The workflow also uses custom tools for code analysis, maintains conversational state and memory, and pauses before producing the final result so that a human can approve the generated solution or request revisions.

The workflow is implemented using LangGraph and LangChain.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
class WorkflowState(TypedDict):
    user_request: str
    generated_code: str
    review: str
    human_feedback: str
    final_output: str

In [ ]:
@tool
def count_code_lines(code: str) -> str:
    """Count the number of non-empty lines in Python code."""
    lines = [line for line in code.splitlines() if line.strip()]
    return f"The generated code contains {len(lines)} non-empty lines."


@tool
def check_python_syntax(code: str) -> str:
    """Check whether Python code has valid syntax."""
    try:
        compile(code, "<string>", "exec")
        return "Syntax check passed: the Python code is syntactically valid."
    except SyntaxError as e:
        return f"Syntax error detected: {e}"

In [ ]:
def code_writer_agent(state: WorkflowState):
    prompt = f"""
You are the Code Writer Agent.

Your responsibility is to create a clear and correct Python solution
for the user's programming request.

User request:
{state['user_request']}

Return ONLY the Python code.
Do not include Markdown code fences.
"""

    response = llm.invoke(prompt)

    code = response.content

    line_info = count_code_lines.invoke({"code": code})

    print("\n--- CODE WRITER AGENT ---")
    print(code)
    print("\nTool result:", line_info)

    return {
        "generated_code": code
    }

In [ ]:
def code_reviewer_agent(state: WorkflowState):

    syntax_result = check_python_syntax.invoke({
        "code": state["generated_code"]
    })

    prompt = f"""
You are the Code Reviewer Agent.

Your responsibility is to review Python code created by another AI agent.

Original request:
{state['user_request']}

Generated code:
{state['generated_code']}

Automated syntax analysis:
{syntax_result}

Review the solution for:
- correctness
- syntax problems
- logical problems
- code quality
- whether it satisfies the user's request

Give a concise review.
"""

    response = llm.invoke(prompt)

    print("\n--- CODE REVIEWER AGENT ---")
    print(response.content)

    print("\nTool result:", syntax_result)

    return {
        "review": response.content
    }

In [ ]:
def human_review_node(state: WorkflowState):
    feedback = interrupt({
        "question": "Do you approve the generated solution?",
        "generated_code": state["generated_code"],
        "review": state["review"]
    })

    return {
        "human_feedback": feedback
    }

In [ ]:
def revision_agent(state: WorkflowState):
    prompt = f"""
You are the Code Writer Agent.

Revise the Python solution based on the human feedback.

Original request:
{state['user_request']}

Current code:
{state['generated_code']}

Reviewer feedback:
{state['review']}

Human feedback:
{state['human_feedback']}

Return ONLY the revised Python code.
Do not include Markdown code fences.
"""

    response = llm.invoke(prompt)

    print("\n--- REVISION ---")
    print(response.content)

    return {
        "generated_code": response.content
    }

In [ ]:
def route_human_feedback(state: WorkflowState):
    feedback = state["human_feedback"].strip().lower()

    if feedback in ["approve", "approved", "yes", "ok"]:
        return "finalize"

    return "revise"

In [ ]:
def finalize_node(state: WorkflowState):
    final_output = f"""
FINAL APPROVED SOLUTION

Code:
{state['generated_code']}

Review:
{state['review']}
"""

    print("\n--- FINAL OUTPUT ---")
    print(final_output)

    return {
        "final_output": final_output
    }

In [ ]:
builder = StateGraph(WorkflowState)

builder.add_node("writer", code_writer_agent)
builder.add_node("reviewer", code_reviewer_agent)
builder.add_node("human_review", human_review_node)
builder.add_node("revision", revision_agent)
builder.add_node("finalize", finalize_node)

In [ ]:
builder.add_edge(START, "writer")
builder.add_edge("writer", "reviewer")
builder.add_edge("reviewer", "human_review")

In [ ]:
builder.add_conditional_edges(
    "human_review",
    route_human_feedback,
    {
        "finalize": "finalize",
        "revise": "revision"
    }
)

In [ ]:
builder.add_edge("revision", "reviewer")
builder.add_edge("finalize", END)

In [ ]:
memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [ ]:
def execute_workflow(user_request: str):
    config = {
        "configurable": {
            "thread_id": "main_thread"
        }
    }

    initial_state = {
        "user_request": user_request,
        "generated_code": "",
        "review": "",
        "human_feedback": "",
        "final_output": ""
    }

    result = graph.invoke(
        initial_state,
        config=config
    )

    return result

In [ ]:
def run_test(user_request: str, thread_id: str):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    initial_state = {
        "user_request": user_request,
        "generated_code": "",
        "review": "",
        "human_feedback": "",
        "final_output": ""
    }

    result = graph.invoke(
        initial_state,
        config=config
    )

    return result, config

# Test 1

In [ ]:
result1, config1 = run_test(
    "Write a Python function that checks whether a number is even.",
    "test_1"
)


--- CODE WRITER AGENT ---
def is_even(number):
    return number % 2 == 0

Tool result: The generated code contains 2 non-empty lines.

--- CODE REVIEWER AGENT ---
### Code Review

**Correctness:**  
The function correctly checks if a number is even by using the modulus operator. If the remainder when dividing by 2 is zero, the number is even.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The function accurately implements the intended functionality.

**Code Quality:**  
The code is concise and clear. The function name `is_even` is descriptive and follows Python naming conventions. However, it could benefit from a type hint for better clarity.

**Satisfaction of User's Request:**  
The function meets the user's request to check whether a number is even.

### Suggestions for Improvement
- Consider adding a type hint for the parameter to indicate that it expects an integer:  
  ```python
  def

In [ ]:
result1 = graph.invoke(
    Command(resume="approve"),
    config=config1
)

print(result1["final_output"])


--- FINAL OUTPUT ---

FINAL APPROVED SOLUTION

Code:
def is_even(number):
    return number % 2 == 0

Review:
### Code Review

**Correctness:**  
The function correctly checks if a number is even by using the modulus operator. If the remainder when dividing by 2 is zero, the number is even.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The function accurately implements the intended functionality.

**Code Quality:**  
The code is concise and clear. The function name `is_even` is descriptive and follows Python naming conventions. However, it could benefit from a type hint for better clarity.

**Satisfaction of User's Request:**  
The function meets the user's request to check whether a number is even.

### Suggestions for Improvement
- Consider adding a type hint for the parameter to indicate that it expects an integer:  
  ```python
  def is_even(number: int) -> bool:
  ```
- Optionally, inc

# Test 2

In [ ]:
result2, config2 = run_test(
    "Write a Python function that returns the largest number in a list.",
    "test_2"
)


--- CODE WRITER AGENT ---
def find_largest_number(numbers):
    if not numbers:
        return None
    largest = numbers[0]
    for number in numbers:
        if number > largest:
            largest = number
    return largest

Tool result: The generated code contains 8 non-empty lines.

--- CODE REVIEWER AGENT ---
### Code Review

**Correctness:**  
The function correctly identifies and returns the largest number in a list. It handles the case where the list is empty by returning `None`, which is appropriate.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The algorithm correctly initializes the largest number and iterates through the list to find the maximum.

**Code Quality:**  
- The code is clear and easy to read. 
- The variable names are descriptive, which enhances readability.
- The function could benefit from a type hint for the parameter to indicate that it expects a list of number

In [ ]:
result2 = graph.invoke(
    Command(resume="Revise the solution so it does not use the built-in max() function."),
    config=config2
)


--- REVISION ---
def find_largest_number(numbers: list[float]) -> float:
    if not numbers:
        raise ValueError("The list is empty. Cannot determine the largest number.")
    largest = numbers[0]
    for number in numbers:
        if number > largest:
            largest = number
    return largest

--- CODE REVIEWER AGENT ---
### Review of the Generated Code

#### Correctness
- The function correctly identifies and returns the largest number in a list of floats. It raises a `ValueError` if the list is empty, which is appropriate.

#### Syntax Problems
- The syntax for type hinting in the function signature is incorrect. Instead of `list[float]`, it should be `List[float]` after importing `List` from the `typing` module. Alternatively, you can use `list` without type hints, but that would lose type specificity.

#### Logical Problems
- There are no logical problems in the implementation. The algorithm correctly initializes the largest number and iterates through the list to find

In [ ]:
result2 = graph.invoke(
    Command(resume="approve"),
    config=config2
)

print(result2["final_output"])


--- FINAL OUTPUT ---

FINAL APPROVED SOLUTION

Code:
def find_largest_number(numbers: list[float]) -> float:
    if not numbers:
        raise ValueError("The list is empty. Cannot determine the largest number.")
    largest = numbers[0]
    for number in numbers:
        if number > largest:
            largest = number
    return largest

Review:
### Review of the Generated Code

#### Correctness
- The function correctly identifies and returns the largest number in a list of floats. It raises a `ValueError` if the list is empty, which is appropriate.

#### Syntax Problems
- The syntax for type hinting in the function signature is incorrect. Instead of `list[float]`, it should be `List[float]` after importing `List` from the `typing` module. Alternatively, you can use `list` without type hints, but that would lose type specificity.

#### Logical Problems
- There are no logical problems in the implementation. The algorithm correctly initializes the largest number and iterates through 

# Test 3

In [ ]:
result3, config3 = run_test(
    "Write a Python function that returns the first n Fibonacci numbers.",
    "test_3"
)


--- CODE WRITER AGENT ---
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1
    for _ in range(n):
        fib_sequence.append(a)
        a, b = b, a + b
    return fib_sequence

Tool result: The generated code contains 7 non-empty lines.

--- CODE REVIEWER AGENT ---
### Code Review

**Correctness:**  
The function correctly generates the first `n` Fibonacci numbers. It initializes the sequence with the first two Fibonacci numbers (0 and 1) and iteratively computes the next numbers in the sequence.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The algorithm correctly follows the Fibonacci sequence generation logic.

**Code Quality:**  
- The code is clear and easy to read. 
- Variable names (`a`, `b`, `fib_sequence`) are appropriate for their purpose.
- The use of a for loop with `_` as the loop variable is a good practice when the variable is not needed.

**Satisfaction of User's Requ

In [ ]:
result3 = graph.invoke(
    Command(resume="approve"),
    config=config3
)

print(result3["final_output"])


--- FINAL OUTPUT ---

FINAL APPROVED SOLUTION

Code:
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1
    for _ in range(n):
        fib_sequence.append(a)
        a, b = b, a + b
    return fib_sequence

Review:
### Code Review

**Correctness:**  
The function correctly generates the first `n` Fibonacci numbers. It initializes the sequence with the first two Fibonacci numbers (0 and 1) and iteratively computes the next numbers in the sequence.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The algorithm correctly follows the Fibonacci sequence generation logic.

**Code Quality:**  
- The code is clear and easy to read. 
- Variable names (`a`, `b`, `fib_sequence`) are appropriate for their purpose.
- The use of a for loop with `_` as the loop variable is a good practice when the variable is not needed.

**Satisfaction of User's Request:**  
The function meets the user's request by retu

# Test 4

In [ ]:
result4, config4 = run_test(
    "Write a Python function that checks whether a string is a palindrome.",
    "test_4"
)


--- CODE WRITER AGENT ---
def is_palindrome(s):
    s = s.lower().replace(" ", "")  # Normalize the string: lower case and remove spaces
    return s == s[::-1]  # Check if the string is equal to its reverse

Tool result: The generated code contains 3 non-empty lines.

--- CODE REVIEWER AGENT ---
### Code Review

**Correctness:**  
The function correctly checks if a string is a palindrome by normalizing the string (converting to lowercase and removing spaces) and comparing it to its reverse. However, it does not handle punctuation or special characters, which may be relevant depending on the definition of a palindrome.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
The logic for checking palindromes is sound, but the normalization step could be improved to handle punctuation and special characters. As it stands, the function may return false negatives for strings like "A man, a plan, a canal, Panama!" which is a palindrome

In [ ]:
result4 = graph.invoke(
    Command(resume="approve"),
    config=config4
)

print(result4["final_output"])


--- FINAL OUTPUT ---

FINAL APPROVED SOLUTION

Code:
def is_palindrome(s):
    s = s.lower().replace(" ", "")  # Normalize the string: lower case and remove spaces
    return s == s[::-1]  # Check if the string is equal to its reverse

Review:
### Code Review

**Correctness:**  
The function correctly checks if a string is a palindrome by normalizing the string (converting to lowercase and removing spaces) and comparing it to its reverse. However, it does not handle punctuation or special characters, which may be relevant depending on the definition of a palindrome.

**Syntax Problems:**  
There are no syntax issues; the code is syntactically valid.

**Logical Problems:**  
The logic for checking palindromes is sound, but the normalization step could be improved to handle punctuation and special characters. As it stands, the function may return false negatives for strings like "A man, a plan, a canal, Panama!" which is a palindrome when ignoring punctuation.

**Code Quality:**  
The c

# Test 5

In [ ]:
result5, config5 = run_test(
    "Write a Python function that removes duplicate values from a list while preserving order.",
    "test_5"
)


--- CODE WRITER AGENT ---
def remove_duplicates(input_list):
    seen = set()
    result = []
    for item in input_list:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

Tool result: The generated code contains 8 non-empty lines.

--- CODE REVIEWER AGENT ---
### Code Review

**Correctness:**  
The function correctly removes duplicate values from the input list while preserving the order of the first occurrences. It uses a set to track seen items, which ensures that each item is only added to the result list once.

**Syntax Problems:**  
There are no syntax problems. The code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The logic for checking duplicates and maintaining order is sound.

**Code Quality:**  
- The code is clean and easy to read. 
- Variable names (`seen` and `result`) are descriptive and appropriate.
- The function is efficient, with a time complexity of O(n), where n is the leng

In [ ]:
result5 = graph.invoke(
    Command(resume="approve"),
    config=config5
)

print(result5["final_output"])


--- FINAL OUTPUT ---

FINAL APPROVED SOLUTION

Code:
def remove_duplicates(input_list):
    seen = set()
    result = []
    for item in input_list:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

Review:
### Code Review

**Correctness:**  
The function correctly removes duplicate values from the input list while preserving the order of the first occurrences. It uses a set to track seen items, which ensures that each item is only added to the result list once.

**Syntax Problems:**  
There are no syntax problems. The code is syntactically valid.

**Logical Problems:**  
There are no logical problems. The logic for checking duplicates and maintaining order is sound.

**Code Quality:**  
- The code is clean and easy to read. 
- Variable names (`seen` and `result`) are descriptive and appropriate.
- The function is efficient, with a time complexity of O(n), where n is the length of the input list.

**Satisfaction of User's Reques